In [ ]:
pip install -U transformers accelerate bitsandbytes sentencepiece seqeval pandas tqdm

In [ ]:
#!/usr/bin/env python3

import os
import re
import time
import warnings

import pandas as pd
import torch

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from seqeval.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report
)


# ============================================================
# CONFIG
# ============================================================

MODEL = "md-nishat-008/TigerLLM-9B-it"

CSV_PATH = "/content/dataset.csv"
OUTPUT_DIR = "/content/tigerllm_9b_8bit_results"

# None = full dataset
# Example: 10, 50, 100
NUM_SAMPLES = None

MAX_INPUT_LENGTH = 2048


# ============================================================
# LOAD TIGERLLM 9B IN 8-BIT
# ============================================================

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL,
    trust_remote_code=True
)


quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)


print("Loading TigerLLM-9B in 8-bit...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True
)

model.eval()

print("Model loaded successfully.")

try:
    print(
        f"Model memory: "
        f"{model.get_memory_footprint() / 1024**3:.2f} GB"
    )
except Exception:
    pass


# ============================================================
# FIND VALID LABELS
# ============================================================

def get_valid_labels(df):

    labels = set()

    for row in df["labels"]:
        for label in str(row).split():
            labels.add(label)

    return sorted(labels)


# ============================================================
# CREATE PROMPT
# ============================================================

def build_prompt(text, valid_labels, expected_count):

    label_text = ", ".join(valid_labels)

    prompt = f"""
You are performing Named Entity Recognition on Bengali healthcare text.

Assign exactly ONE NER label to each whitespace-separated word.

Valid labels:
{label_text}

Sentence:
{text}

Number of words:
{expected_count}

Rules:
1. Return exactly {expected_count} labels.
2. Keep the same order as the words.
3. Use only labels from the valid label list.
4. Separate labels using spaces.
5. Do not explain your answer.
6. Do not output the sentence.
7. Output only the labels.

Answer:
""".strip()

    return prompt


# ============================================================
# CLEAN MODEL OUTPUT
# ============================================================

def extract_labels(output, valid_labels, expected_count):

    output = output.strip()

    # Remove markdown/code formatting
    output = output.replace("```", " ")

    valid_set = set(valid_labels)

    # Split output
    parts = re.split(r"[\s,\n]+", output)

    predicted = []

    for part in parts:

        part = part.strip()

        # Remove common punctuation
        part = part.strip("[](){}\"'.,:;")

        if part in valid_set:
            predicted.append(part)

    # Too many labels -> truncate
    predicted = predicted[:expected_count]

    # Too few labels -> pad with O
    if len(predicted) < expected_count:

        pad_label = "O" if "O" in valid_set else valid_labels[0]

        predicted.extend(
            [pad_label] * (expected_count - len(predicted))
        )

    return predicted


# ============================================================
# GENERATE TIGERLLM PREDICTION
# ============================================================

def predict(text, valid_labels, expected_count):

    prompt = build_prompt(
        text,
        valid_labels,
        expected_count
    )

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    try:

        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    except Exception:

        formatted_prompt = prompt


    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    )

    input_device = next(model.parameters()).device

    inputs = {
        key: value.to(input_device)
        for key, value in inputs.items()
    }


    # NER label sequences can require several subword tokens
    max_new_tokens = min(
        512,
        max(32, expected_count * 5)
    )


    start_time = time.time()

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    latency = time.time() - start_time


    # Decode ONLY newly generated tokens
    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    output_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    labels = extract_labels(
        output_text,
        valid_labels,
        expected_count
    )

    return labels, output_text, latency


# ============================================================
# RUN DATASET
# ============================================================

def run_model_on_dataset(model_name, df, valid_labels, output_dir):

    rows = []

    for index, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=model_name
    ):

        text = str(row["text"]).strip()

        true_labels = str(row["labels"]).strip().split()

        expected_count = len(true_labels)

        try:

            predicted_labels, raw_output, latency = predict(
                text,
                valid_labels,
                expected_count
            )

            rows.append({
                "index": index,
                "text": text,
                "true_labels": " ".join(true_labels),
                "pred_labels": " ".join(predicted_labels),
                "raw_output": raw_output,
                "latency": latency,
                "error": None
            })

        except Exception as e:

            print(
                f"\nError at row {index}: {e}"
            )

            rows.append({
                "index": index,
                "text": text,
                "true_labels": " ".join(true_labels),
                "pred_labels": "",
                "raw_output": "",
                "latency": None,
                "error": str(e)
            })


        # Save continuously
        pd.DataFrame(rows).to_csv(
            os.path.join(
                output_dir,
                "predictions_checkpoint.csv"
            ),
            index=False,
            encoding="utf-8-sig"
        )

    return rows


# ============================================================
# COMPUTE METRICS
# ============================================================

def compute_metrics(rows):

    y_true = []
    y_pred = []

    all_true_flat = []
    all_pred_flat = []

    latencies = []


    for r in rows:

        if (
            pd.notna(r.get("true_labels"))
            and
            pd.notna(r.get("pred_labels"))
            and
            str(r.get("pred_labels")).strip()
        ):

            t = str(r["true_labels"]).split()

            p = str(r["pred_labels"]).split()

            # Predictions should already have equal length,
            # but this protects against malformed output
            min_len = min(len(t), len(p))

            t = t[:min_len]
            p = p[:min_len]

            if len(t) > 0:

                y_true.append(t)
                y_pred.append(p)

                all_true_flat.extend(t)
                all_pred_flat.extend(p)


        if r.get("latency") is not None:

            latencies.append(
                r["latency"]
            )


    correct = sum(
        t == p
        for t, p in zip(
            all_true_flat,
            all_pred_flat
        )
    )


    token_accuracy = (
        correct / len(all_true_flat)
        if all_true_flat
        else 0
    )


    return {

        "token_accuracy":
            token_accuracy,

        "f1":
            f1_score(
                y_true,
                y_pred,
                average="weighted"
            ),

        "precision":
            precision_score(
                y_true,
                y_pred,
                average="weighted"
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                average="weighted"
            ),

        "total":
            len(rows),

        "avg_latency":
            (
                sum(latencies) / len(latencies)
                if latencies
                else None
            ),

        "report":
            classification_report(
                y_true,
                y_pred,
                digits=4
            )
    }


# ============================================================
# MAIN
# ============================================================

def main():

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )


    df = pd.read_csv(
        CSV_PATH
    )


    if not {
        "text",
        "labels"
    }.issubset(df.columns):

        raise ValueError(
            "CSV must contain 'text' and 'labels' columns"
        )


    if NUM_SAMPLES is not None:

        df = df.head(
            NUM_SAMPLES
        ).reset_index(
            drop=True
        )


    print(
        f"\nDataset: {len(df)} sentences"
    )


    # Find all NER labels
    valid_labels = get_valid_labels(df)

    print("\nValid labels:")

    for label in valid_labels:
        print(" ", label)


    # Label distribution
    from collections import Counter

    all_labels = [
        lbl
        for row in df["labels"]
        for lbl in str(row).split()
    ]


    print("\nLabel distribution:")

    for lbl, cnt in sorted(
        Counter(all_labels).items()
    ):

        print(
            f"  {lbl:<30} {cnt}"
        )


    # Run TigerLLM
    rows = run_model_on_dataset(
        MODEL,
        df,
        valid_labels,
        OUTPUT_DIR
    )


    # Metrics
    metrics = compute_metrics(
        rows
    )


    # Save predictions
    pd.DataFrame(
        rows
    ).to_csv(

        os.path.join(
            OUTPUT_DIR,
            "predictions.csv"
        ),

        index=False,
        encoding="utf-8-sig"
    )


    # Overall benchmark results
    results_df = pd.DataFrame([{

        "model":
            MODEL,

        "quantization":
            "8-bit",

        "token_accuracy":
            metrics["token_accuracy"],

        "f1_weighted":
            metrics["f1"],

        "precision_weighted":
            metrics["precision"],

        "recall_weighted":
            metrics["recall"],

        "total_sentences":
            metrics["total"],

        "avg_latency_s":
            metrics["avg_latency"]
    }])


    results_df.to_csv(

        os.path.join(
            OUTPUT_DIR,
            "benchmark_results.csv"
        ),

        index=False,
        encoding="utf-8-sig"
    )


    print(
        "\n── Overall Results ──"
    )

    print(
        results_df.to_string(
            index=False
        )
    )


    print(
        "\n── seqeval Span-Level Report "
        "(per entity type) ──"
    )

    print(
        metrics["report"]
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()